#  Index same collection with different settings. Check their size and see how they differ in Luke.

In [3]:
import pandas
df = pandas.read_csv("/mnt/wiki_movie_plots_deduped.csv", sep=',')

In [4]:
import lucene
lucene.initVM()

Oct 03, 2025 11:19:21 AM org.apache.lucene.internal.vectorization.PanamaVectorizationProvider <init>
INFO: Java vector incubator API enabled; uses preferredBitSize=512; FMA enabled


In [ ]:
from org.apache.lucene.store import FSDirectory
from java.nio.file import Paths

indexPath = "/mnt/movie_index_2"
indexDir = FSDirectory.open(Paths.get(indexPath))

In [ ]:
from org.apache.lucene.analysis.en import EnglishAnalyzer
from org.apache.lucene.index import IndexWriter, IndexWriterConfig

analyzer = EnglishAnalyzer()
config = IndexWriterConfig(analyzer)
writer = IndexWriter(indexDir, config)

In [ ]:
from org.apache.lucene.document import Document, IntField, TextField, StringField, StoredField, Field, FieldType
from org.apache.lucene.index import IndexOptions

indexType = FieldType()
indexType.setStored(False)
indexType.setTokenized(True)
indexType.setStoreTermVectors(True)
indexType.setIndexOptions(IndexOptions.DOCS_AND_FREQS_AND_POSITIONS)
indexType.setStoreTermVectorPositions(True)
indexType.setStoreTermVectorOffsets(False)

def indexDocument(movie):

    doc = Document() # one Lucene document

    # 1. Year
    year = int(movie['Release Year'])
    doc.add(IntField("YEAR", year, Field.Store.YES))

    # 2. Title
    title = movie['Title']
    doc.add(TextField("TITLE", title, Field.Store.YES))

    # 3. Origin
    origin = movie['Origin/Ethnicity']
    doc.add(StringField("ORIGIN", origin, Field.Store.NO))

    # 4. Director
    director = movie['Director']
    doc.add(TextField("DIRECTOR", director, Field.Store.YES))

    # 5. Cast
    cast = str(movie['Cast'])
    doc.add(TextField("CAST", cast, Field.Store.YES))

    # 6. Genre
    genre = movie['Genre']
    doc.add(StringField("GENRE", genre, Field.Store.YES))

    # 7. Wiki page
    wiki = movie['Wiki Page']
    doc.add(StoredField("URL", wiki))
    
    # 8. Plot
    plot = movie['Plot']
    doc.add(Field("PLOT", plot, indexType))
    
    return doc

In [ ]:
docid = 1
for _, row in df.iterrows():
    print("{} - {}".format(docid, row['Title']))
    docid += 1
    doc = indexDocument(row)
    writer.addDocument(doc)

writer.close()

# Print the stored values in an index (Stored vs. Unstored).

In [3]:
import lucene
import os
from java.nio.file import Paths
from org.apache.lucene.index import DirectoryReader
from org.apache.lucene.store import FSDirectory

# Initialize JVM
lucene.initVM()

# Path for index directory (creates folder if not exists)
# index_path = "lucene_index"
index_path = "/mnt/movie_index_2"

# Open FSDirectory (on-disk index)
directory = FSDirectory.open(Paths.get(index_path))

reader = DirectoryReader.open(directory)

# number of documents in the collection
number_of_documents = reader.numDocs()
print("The number of documents in the index:", number_of_documents)

Oct 03, 2025 11:44:54 AM org.apache.lucene.internal.vectorization.PanamaVectorizationProvider <init>
INFO: Java vector incubator API enabled; uses preferredBitSize=512; FMA enabled


The number of documents in the index: 34886


In [6]:
for i in range(number_of_documents):
    d = reader.storedFields().document(i)
    print(i, "Title:", d.get('TITLE'), end="\t")
    print("URL:", d.get("URL"))
    print("Plot: ", d.get('PLOT'))



0 Title: Kansas Saloon Smashers	URL: https://en.wikipedia.org/wiki/Kansas_Saloon_Smashers
Plot:  None
1 Title: Love by the Light of the Moon	URL: https://en.wikipedia.org/wiki/Love_by_the_Light_of_the_Moon
Plot:  None
2 Title: The Martyred Presidents	URL: https://en.wikipedia.org/wiki/The_Martyred_Presidents
Plot:  None
3 Title: Terrible Teddy, the Grizzly King	URL: https://en.wikipedia.org/wiki/Terrible_Teddy,_the_Grizzly_King
Plot:  None
4 Title: Jack and the Beanstalk	URL: https://en.wikipedia.org/wiki/Jack_and_the_Beanstalk_(1902_film)
Plot:  None
5 Title: Alice in Wonderland	URL: https://en.wikipedia.org/wiki/Alice_in_Wonderland_(1903_film)
Plot:  None
6 Title: The Great Train Robbery	URL: https://en.wikipedia.org/wiki/The_Great_Train_Robbery_(1903_film)
Plot:  None
7 Title: The Suburbanite	URL: https://en.wikipedia.org/wiki/The_Suburbanite
Plot:  None
8 Title: The Little Train Robbery	URL: https://en.wikipedia.org/wiki/The_Little_Train_Robbery
Plot:  None
9 Title: The Night Befor

# Print the stored values of all the documents

# Apply different analyzers in different fields of an index.

In [ ]:
import lucene, os
from java.nio.file import Paths
from org.apache.lucene.store import FSDirectory
from org.apache.lucene.analysis.standard import StandardAnalyzer
from org.apache.lucene.analysis.en import EnglishAnalyzer
from org.apache.lucene.analysis.de import GermanAnalyzer
from org.apache.lucene.index import IndexWriter, IndexWriterConfig
from org.apache.lucene.document import Document, StringField, TextField, Field

# --- Initialize JVM ---
if not lucene.getVMEnv()
    lucene.initVM()

# --- Setup index directory ---
index_path = "/mnt/index_per_field_2"
directory = FSDirectory.open(Paths.get(index_path))

# --- Define analyzers ---
english_analyzer = EnglishAnalyzer()
german_analyzer  = GermanAnalyzer()


ValueError: JVM is already running, options are ineffective

In [2]:
# --- Sample documents ---
docs = [
    {
        "docID": "1",
        "title": "Sample document",
        "content_en": "This is an English example sentence with many words",
        "content_de": "Das ist ein deutscher Beispielsatz mit vielen Wörtern"
    },
    {
        "docID": "2",
        "title": "Document on Lucene",
        "content_en": "Lucene is a very powerful search library",
        "content_de": "Lucene ist eine sehr leistungsfähige Suchbibliothek"
    }
]

# Initialize per-field analyzer mapping
analyzer_map = {"content_en": english_analyzer,
                "content_de": german_analyzer
                }

from java.util import HashMap

analyzer_map_java = HashMap()
analyzer_map_java.put("content_en", english_analyzer)
analyzer_map_java.put("content_de", german_analyzer)

# Construct PerFieldAnalyzerWrapper
from org.apache.lucene.analysis.miscellaneous import PerFieldAnalyzerWrapper
per_field_analyzer = PerFieldAnalyzerWrapper(StandardAnalyzer(), analyzer_map_java)

# Init IndexWriterConfig
config = IndexWriterConfig(per_field_analyzer)
writer = IndexWriter(directory, config)



In [ ]:
from org.apache.lucene.document import Document, IntField, TextField, StringField, StoredField, Field, FieldType
from org.apache.lucene.index import IndexOptions

indexType = FieldType()
indexType.setStored(True)
indexType.setTokenized(True)
indexType.setStoreTermVectors(True)
indexType.setIndexOptions(IndexOptions.DOCS_AND_FREQS_AND_POSITIONS)
indexType.setStoreTermVectorPositions(True)
indexType.setStoreTermVectorOffsets(False)

# --- Add docs ---
for d in docs:
    doc = Document()
    doc.add(StringField("docID", d['docID'], Field.Store.YES))
    doc.add(TextField("title", d['title'], Field.Store.YES))
    doc.add(Field("content_en", d['content_en'], indexType))
    doc.add(Field("content_de", d['content_de'], indexType))
    writer.addDocument(doc)

writer.close()
print("Index created at:", index_path)


Index created at: /mnt/index_per_field


#  Index Indian language text collection.

In [1]:
import pandas as pd
from io import StringIO

# df = pd.read_csv("/mnt/datasets/test.csv")  # Error due to wrong formatting

with open("/mnt/datasets/test.csv", 'r') as f:
    data = f.read().replace('\n ', ' ')

df = pd.read_csv(StringIO(data))

df = df.dropna()

In [2]:
import lucene

if not lucene.getVMEnv():
    lucene.initVM()

Oct 03, 2025 6:17:14 PM org.apache.lucene.internal.vectorization.PanamaVectorizationProvider <init>
INFO: Java vector incubator API enabled; uses preferredBitSize=512; FMA enabled


In [5]:
from org.apache.lucene.index import IndexWriter, IndexWriterConfig
from org.apache.lucene.store import FSDirectory
from java.nio.file import Paths
from org.apache.lucene.document import Document, IntField, TextField, StringField, StoredField, Field, FieldType
from org.apache.lucene.index import IndexOptions
from org.apache.lucene.analysis.hi import HindiAnalyzer

# setting the index directory
indexPath = "/mnt/hindi_index_2"
indexDir = FSDirectory.open(Paths.get(indexPath))

# setting the analyzer
analyzer = HindiAnalyzer()
config = IndexWriterConfig(analyzer)

# setting the IndexWriter
writer = IndexWriter(indexDir, config)


In [6]:
docid = 0

indexType = FieldType()
indexType.setStored(True)
indexType.setTokenized(True)
indexType.setStoreTermVectors(True)
indexType.setIndexOptions(IndexOptions.DOCS_AND_FREQS_AND_POSITIONS)
indexType.setStoreTermVectorPositions(True)
indexType.setStoreTermVectorOffsets(False)

for _, row in df.iterrows():
    docid += 1

    h = row['headline']
    s = row['summary']
    a = row['article']

    print(docid, " - ", h)
    doc = Document()

    # initialize the fields of the document
    doc.add(TextField("HEADLINE", h, Field.Store.YES))
    doc.add(TextField("SUMMARY", s, Field.Store.YES))
    doc.add(Field("ARTICLE", a, indexType))

    writer.addDocument(doc)

print("Indexing completed. {} documents indexed".format(docid))

writer.close()

1  -  पाकिस्तानी जासूस को मिली अहम खुफिया जानकारी, जम्मू कश्मीर के पुलिस अधिकारी से हुई लापरवाही  
2  -  अमर को अस्पताल से छुट्टी, बोले, सपा से अब नाम न जोड़ें  
3  -  सैलरी नेगोश‍िएट करते वक्‍त भूलकर भी न कहें ये 6 बातें  
4  -  इंटरनेट से पता कीजिए कहां तक पहुंची ट्रेन  
5  -  जब फिल्मी अंदाज में बोले शिवपाल - मुलायम जहां खड़े हो जाते हैं, वहीं से होती है सपा की शुरुआत  
6  -  पाकिस्तान के साथ क्रिकेट खेलने का सवाल ही पैदा नहीं होता : अनुराग ठाकुर  
7  -  मुजफ्फरनगर हिंसा : संगीत सोम सहित तीन आरोपी विधायक गिरफ्तार  
8  -  अयोध्या में भव्य तरीके से मनाया जाएगा दीपोत्सव, 4 लाख दीये जलाकर विश्व रिकॉर्ड बनाने की तैयारी  
9  -  Bigg Boss 11: अर्शी ने सलमान को लिया लपेटे में, शिल्पा की साइड लेने का लगाया आरोप  
10  -  सारा अली खान मम्मी के साथ पहुंचीं डोसा खाने तो पूछ बैठीं ऐसा सवाल, अमृता सिंह ने छिपाया मुंह- देखें Video  
11  -  बॉलीवुड के मशहूर फिल्म निर्माता जे ओम प्रकाश का निधन, नाना को अपना गुरु मानते थे ऋतिक रोशन  
12  -  सोनू निगम के बाद अब सुचित्रा कृष्णमूर्ति ने अजान को लेकर शिकाय